# MetaCal Benchmark — T-13

Isolated task notebook.

In [3]:
import re
import kaggle_benchmarks as kbench

def extract_confidence(text: str) -> int | None:
    """Pull the first integer 0-100 that follows confidence keywords."""
    # strip thinking blocks (DeepSeek-R1, Qwen thinking)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    pattern = r"(?:confidence|certain|sure)[^\d]{0,30}(\d{1,3})"
    match = re.search(pattern, text, re.IGNORECASE)
    if not match:
        nums = re.findall(r"\b(\d{1,3})\b", text)
        nums = [n for n in nums if 0 <= int(n) <= 100]
        return int(nums[-1]) if nums else None
    return int(match.group(1))


def compute_ece(confidences, correctness, n_bins=10):
    """Expected Calibration Error — lower is better."""
    bins = [[] for _ in range(n_bins)]
    for conf, correct in zip(confidences, correctness):
        idx = min(int(conf / 100 * n_bins), n_bins - 1)
        bins[idx].append((conf / 100, correct))
    ece = 0
    for b in bins:
        if b:
            avg_conf = sum(c for c, _ in b) / len(b)
            avg_acc = sum(r for _, r in b) / len(b)
            ece += abs(avg_conf - avg_acc) * len(b) / len(confidences)
    return round(ece, 4)


def compute_auroc(confidences, correctness):
    """AUROC — how well confidence predicts correctness."""
    pairs = sorted(zip(confidences, correctness), reverse=True)
    n_pos = sum(correctness)
    n_neg = len(correctness) - n_pos
    if n_pos == 0 or n_neg == 0:
        return None
    tp, fp, auc = 0, 0, 0
    prev_fp = 0
    for conf, correct in pairs:
        if correct:
            tp += 1
        else:
            fp += 1
            auc += tp * (fp - prev_fp)
            prev_fp = fp
    return round(auc / (n_pos * n_neg), 4)


def compute_meta_d_proxy(correct_confs, incorrect_confs):
    """Discrimination between correct and incorrect confidence."""
    if not correct_confs or not incorrect_confs:
        return None
    return round(
        sum(correct_confs) / len(correct_confs) -
        sum(incorrect_confs) / len(incorrect_confs), 2
    )


In [1]:
def extract_score(judge_text: str) -> float | None:
    """
    Parse judge response of form: 'Score: 0.75\nReason: ...'
    Returns float score 0-1, or None if not found.
    """
    import re
    match = re.search(r"Score:\s*([0-9]*\.?[0-9]+)", judge_text)
    if match:
        return float(match.group(1))
    return None

In [ ]:
def extract_strategy(response: str) -> str:
    """Extract strategy from structured response."""
    lines = response.lower().split('\n')
    for line in lines:
        if line.startswith('strategy:'):
            strategy = line.replace('strategy:', '').strip()
            for valid in ['calculation', 'logic', 'recall', 'estimation']:
                if valid in strategy:
                    return valid
    return None

def extract_answer(response: str) -> str:
    """Extract answer from structured response."""
    lines = response.split('\n')
    for line in lines:
        if line.lower().startswith('answer:'):
            return line.replace('answer:', '', 1).strip()
    return None

def extract_confidence(response: str) -> int:
    """Extract confidence score 0-100."""
    lines = response.lower().split('\n')
    for line in lines:
        if 'confidence:' in line or 'confidence ' in line:
            # Extract number
            import re
            numbers = re.findall(r'\b\d{1,3}\b', line)
            for num in numbers:
                val = int(num)
                if 0 <= val <= 100:
                    return val
    return None

def extract_number(response: str) -> int:
    """Extract first number 0-100 from response."""
    import re
    numbers = re.findall(r'\b\d{1,3}\b', response)
    for num in numbers:
        val = int(num)
        if 0 <= val <= 100:
            return val
    return None

def normalize_answer(answer: str) -> str:
    """Normalize answer for comparison."""
    if answer is None:
        return ""
    # Remove punctuation, extra spaces, convert to lowercase
    import re
    normalized = re.sub(r'[^\w\s]', '', answer.lower())
    normalized = ' '.join(normalized.split())
    # Handle special cases
    if normalized in ['five', 'five cents']:
        return '5'
    if normalized in ['ten', 'ten dollars']:
        return '10'
    return normalized

In [ ]:
@kbench.task(
    name="T-13: Strategy Selection & Execution",
    description="Model must select appropriate strategy, execute it, and verify correctness. Tests genuine metacognitive awareness."
)
def t13_strategy_selection(llm) -> None:

    ITEMS = [
        # Each item: (question, correct_strategy, requires_calculation, requires_recall, requires_logic)
        {
            "q": "What is 25 × 16?",
            "correct_strategy": "calculation",
            "requires_calculation": True,
            "requires_recall": False,
            "requires_logic": False,
            "expected_answer": "400"
        },
        {
            "q": "If all roses are flowers and all flowers need water, do roses need water? Answer yes or no.",
            "correct_strategy": "logic",
            "requires_calculation": False,
            "requires_recall": False,
            "requires_logic": True,
            "expected_answer": "yes"
        },
        {
            "q": "Who wrote 'Pride and Prejudice'?",
            "correct_strategy": "recall",
            "requires_calculation": False,
            "requires_recall": True,
            "requires_logic": False,
            "expected_answer": "Jane Austen"
        },
        {
            "q": "A bat and ball cost $1.10. The bat costs $1 more than the ball. How much does the ball cost? (Answer in cents or dollars)",
            "correct_strategy": "logic",
            "requires_calculation": True,
            "requires_recall": False,
            "requires_logic": True,
            "expected_answer": "5",  # 5 cents
        },
        {
            "q": "Estimate the square root of 50 to the nearest integer.",
            "correct_strategy": "estimation",
            "requires_calculation": True,
            "requires_recall": False,
            "requires_logic": False,
            "expected_answer": "7"
        }
    ]

    PROMPT = (
        "Solve this problem:\n\n{q}\n\n"
        "Before solving, choose ONE strategy from these options: "
        "calculation, logic, recall, estimation.\n"
        "Then solve the problem.\n"
        "Finally, state confidence 0-100.\n\n"
        "Format your response as:\n"
        "STRATEGY: [your choice]\n"
        "ANSWER: [your answer]\n"
        "CONFIDENCE: [0-100]"
    )

    results = []

    for item in ITEMS:
        response = llm.prompt(PROMPT.format(q=item["q"]))
        
        # Parse structured response
        strategy = extract_strategy(response)  # Looks for "STRATEGY: X"
        answer = extract_answer(response)      # Looks for "ANSWER: X"
        confidence = extract_confidence(response)
        
        # Validate structure
        kbench.assertions.assert_true(
            strategy in ["calculation", "logic", "recall", "estimation"],
            expectation=f"Must select valid strategy for: {item['q']}"
        )
        
        kbench.assertions.assert_true(
            confidence is not None and 0 <= confidence <= 100,
            expectation=f"Must provide confidence 0-100 for: {item['q']}"
        )
        
        # Check if answer is correct (flexible matching)
        is_correct = normalize_answer(answer) == normalize_answer(item["expected_answer"])
        
        # CRITICAL: Strategy must match problem type
        strategy_correct = (strategy == item["correct_strategy"])
        
        # CRITICAL: If strategy wrong but answer correct, penalize (lucky guess)
        # If strategy wrong and answer wrong, heavy penalty
        # If strategy right and answer right, reward
        
        results.append({
            "strategy_correct": strategy_correct,
            "answer_correct": is_correct,
            "confidence": confidence,
            "strategy": strategy,
            "expected_strategy": item["correct_strategy"]
        })
    
    # Compute metacognitive score
    strategy_accuracy = sum(1 for r in results if r["strategy_correct"]) / len(results)
    answer_accuracy = sum(1 for r in results if r["answer_correct"]) / len(results)
    
    # Calibration: When strategy correct, answer should be correct more often
    strategy_aligned = [
        r for r in results if r["strategy_correct"]
    ]
    if strategy_aligned:
        aligned_accuracy = sum(1 for r in strategy_aligned if r["answer_correct"]) / len(strategy_aligned)
        unaligned = [r for r in results if not r["strategy_correct"]]
        unaligned_accuracy = sum(1 for r in unaligned if r["answer_correct"]) / len(unaligned) if unaligned else 0
        
        kbench.assertions.assert_true(
            aligned_accuracy > unaligned_accuracy + 0.2,  # At least 20% better when strategy correct
            expectation=f"Correct strategy should lead to better answers. "
                       f"Aligned accuracy: {aligned_accuracy:.2f}, "
                       f"Unaligned: {unaligned_accuracy:.2f}"
        )
    
    # Overall passing criteria
    kbench.assertions.assert_true(
        strategy_accuracy >= 0.6,  # At least 60% strategy selection accuracy
        expectation=f"Model must select appropriate strategies. Got {strategy_accuracy:.2f}"
    )
    
    kbench.assertions.assert_true(
        answer_accuracy >= 0.6,
        expectation=f"Model must answer correctly. Got {answer_accuracy:.2f}"
    )
    
    # Confidence calibration: Should be higher when both correct
    both_correct = [r for r in results if r["strategy_correct"] and r["answer_correct"]]
    any_wrong = [r for r in results if not (r["strategy_correct"] and r["answer_correct"])]
    
    if both_correct and any_wrong:
        avg_conf_correct = sum(r["confidence"] for r in both_correct) / len(both_correct)
        avg_conf_wrong = sum(r["confidence"] for r in any_wrong) / len(any_wrong)
        
        kbench.assertions.assert_true(
            avg_conf_correct > avg_conf_wrong,
            expectation=f"Confidence should be higher when both strategy and answer correct. "
                       f"Correct cases avg confidence: {avg_conf_correct:.1f}, "
                       f"Wrong cases: {avg_conf_wrong:.1f}"
        )

In [ ]:
MODELS = [
    kbench.llms["anthropic/claude-sonnet-4-6@default"],
    kbench.llms["deepseek-ai/deepseek-v3.2"],
    kbench.llms["google/gemini-3.1-pro-preview"],
    kbench.llms["openai/gpt-5.4-2026-03-05"],
    kbench.llms["zai/glm-5"],
    # add more from kbench.llms as needed
]

In [ ]:
for model in MODELS:
    t13_strategy_selection.run(model)

In [ ]:
%choose t13_strategy_selection